### 승인/거부

> https://docs.langchain.com/oss/python/langgraph/interrupts#approve-or-reject

In [ ]:
from typing import Literal, Optional, TypedDict

class ApprovalState(TypedDict):
    action_details: str
    # status는 '보류', '승인', '거부' 중 하나일 수 있습니다.
    status: Optional[Literal["보류", "승인", "거부"]]


In [17]:
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command, interrupt

def approval_node(state: ApprovalState) -> Command[Literal["proceed", "cancel"]]:
    # 사용자에게 승인 또는 거부를 묻기 위해 interrupt 호출
    # 사용자에게 보여 줄 메시지로 dict 전달
    decision = interrupt({
        "question": "이 작업을 승인하시겠습니까?",
        "details": state["action_details"],
    })

    # 사용자의 결정을 바탕으로 다음 Node로 이동
    return Command(goto="proceed" if decision else "cancel")


def proceed_node(state: ApprovalState):
    return {"status": "승인"}


def cancel_node(state: ApprovalState):
    return {"status": "거부"}

In [18]:
from langgraph.graph import StateGraph, START, END

graph_builder = StateGraph(ApprovalState)

graph_builder.add_node("approval", approval_node)
graph_builder.add_node("proceed", proceed_node)
graph_builder.add_node("cancel", cancel_node)

graph_builder.add_edge(START, "approval")
graph_builder.add_edge("proceed", END)
graph_builder.add_edge("cancel", END)

checkpointer = InMemorySaver()
graph = graph_builder.compile(checkpointer=checkpointer)

In [46]:
config = {"configurable": {"thread_id": "approval-1"}}

In [47]:
initial = graph.invoke(
    {"action_details": "5만원을 송금해 줘", "status": "보류"},
    config=config,
)

In [48]:
initial["__interrupt__"]

[Interrupt(value={'question': '이 작업을 승인하시겠습니까?', 'details': '5만원을 송금해 줘'}, id='99277b1810d31a9e9cc96f6f2bfc9b16')]

In [22]:
# 사용자 입력에 따라 재개 (True: 승인, False: 거부)
# 승인을 선택
resumed = graph.invoke(Command(resume=True), config=config)

In [23]:
resumed

{'action_details': '5만원을 송금해 줘', 'status': '승인'}

In [24]:
config = {"configurable": {"thread_id": "approval-2"}}

In [25]:
initial = graph.invoke(
    {"action_details": "5만원을 송금해 줘", "status": "보류"},
    config=config,
)

In [26]:
initial["__interrupt__"]

[Interrupt(value={'question': '이 작업을 승인하시겠습니까?', 'details': '5만원을 송금해 줘'}, id='1217e5c294d70132b1dd903abc5ca65b')]

In [28]:
state = graph.get_state(config)

In [29]:
state

StateSnapshot(values={'action_details': '5만원을 송금해 줘', 'status': '보류'}, next=('approval',), config={'configurable': {'thread_id': 'approval-2', 'checkpoint_ns': '', 'checkpoint_id': '1f10d354-95e6-68c9-8000-80326a58e191'}}, metadata={'source': 'loop', 'step': 0, 'parents': {}}, created_at='2026-02-19T01:49:54.054271+00:00', parent_config={'configurable': {'thread_id': 'approval-2', 'checkpoint_ns': '', 'checkpoint_id': '1f10d354-95e4-605d-bfff-197ca590c3a3'}}, tasks=(PregelTask(id='482fab15-0507-78fd-bcf9-7da507e8eab5', name='approval', path=('__pregel_pull', 'approval'), error=None, interrupts=(Interrupt(value={'question': '이 작업을 승인하시겠습니까?', 'details': '5만원을 송금해 줘'}, id='1217e5c294d70132b1dd903abc5ca65b'),), state=None, result=None),), interrupts=(Interrupt(value={'question': '이 작업을 승인하시겠습니까?', 'details': '5만원을 송금해 줘'}, id='1217e5c294d70132b1dd903abc5ca65b'),))

In [49]:
state.interrupts[0].value

{'question': '이 작업을 승인하시겠습니까?', 'details': '5만원을 송금해 줘'}

In [50]:
prompt = state.interrupts[0].value['question']
user_input = True if input(prompt).lower() == 'true' else False # 재개(resume)를 위해 사용자 입력 받기

In [51]:
user_input

False

In [ ]:
resumed = graph.invoke(Command(resume=user_input), config=config)

In [53]:
resumed

{'action_details': '5만원을 송금해 줘', 'status': '거부'}